# 01 – Exploring the match data

Week 1 deliverable: get to know the 12 seasons (2014/15–2025/26) before building features.

The clean table is built by `src/data_loader.py`. If `data/processed/matches.csv` doesn't exist yet, run this from the repo root:

```
python src/data_loader.py
```

**Questions for this notebook**
1. How often do home wins, draws and away wins happen?
2. Has home advantage changed over time?
3. How many goals are there per game?

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from data_loader import load_matches

# Chart style: light surface, muted axes, hairline grid.
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e",
    "xtick.color": "#898781", "ytick.color": "#898781",
    "axes.grid": True, "axes.grid.axis": "y", "axes.axisbelow": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "figure.figsize": (10, 4.5),
})
# One fixed colour per outcome, used in every chart.
COLORS = {"H": "#2a78d6", "D": "#eb6834", "A": "#1baf7a"}
NAMES = {"H": "Home win", "D": "Draw", "A": "Away win"}

## 1. Load and inspect

In [ ]:
matches = load_matches()
print(matches.shape)
matches.head()

In [ ]:
matches.info()

**Pandas basics used here:**

- `df["col"]` selects one column; `df[["a", "b"]]` selects several.
- `df[df["season"] == "2020/21"]` filters rows with a boolean condition.
- `.value_counts()` counts each distinct value.
- `.groupby("season")` splits the table into groups, then `.mean()`, `.size()` etc. summarise each group.

In [ ]:
# Every season should have 380 matches (20 teams x 19 opponents x home and away).
matches["season"].value_counts().sort_index()

In [ ]:
# Team names must be spelled the same way in every season, or rolling form in Week 2 breaks.
# Count seasons per team: 12 = ever-present, low numbers = promoted/relegated clubs.
team_seasons = (
    pd.concat([matches[["season", "home_team"]].rename(columns={"home_team": "team"}),
               matches[["season", "away_team"]].rename(columns={"away_team": "team"})])
    .drop_duplicates()
    .groupby("team")["season"].count()
    .sort_values(ascending=False)
)
print(f"{len(team_seasons)} different teams")
team_seasons

In [ ]:
# Missing values: only Pinnacle odds are missing (part of 2025/26). Use Bet365 or market-average odds in Week 4.
missing = matches.isna().sum()
missing[missing > 0]

## 2. Home win / draw / away win

In [ ]:
overall = matches["result"].value_counts(normalize=True).reindex(["H", "D", "A"])
overall.rename(NAMES).mul(100).round(1)

This is **Baseline 2** for Week 3: if you know nothing about the teams, predict these percentages for every match.
**Baseline 1** ("always pick the home team") would be right about as often as the home-win percentage above.

In [ ]:
by_season = (
    matches.groupby("season")["result"]
    .value_counts(normalize=True)
    .unstack()[["H", "D", "A"]]
    .mul(100)
)
by_season.round(1)

In [ ]:
fig, ax = plt.subplots()
for r in ["H", "D", "A"]:
    ax.plot(by_season.index, by_season[r], color=COLORS[r], linewidth=2, marker="o", markersize=6, label=NAMES[r])
    ax.annotate(NAMES[r], (len(by_season) - 1, by_season[r].iloc[-1]), xytext=(8, 0),
                textcoords="offset points", va="center", color="#52514e", fontsize=10)
ax.set_title("Share of results per season")
ax.set_ylabel("% of matches")
ax.set_ylim(0, 55)
ax.legend(loc="upper left", ncol=3, frameon=False)
ax.tick_params(axis="x", rotation=45)
ax.set_xlim(-0.5, len(by_season) + 0.8)
plt.tight_layout()
plt.show()

## 3. Home advantage over time

Here *home advantage* means home win % minus away win %.
2020/21 was played almost entirely without fans (COVID), so it's a natural experiment: does home advantage shrink without a crowd?

In [ ]:
home_adv = (by_season["H"] - by_season["A"]).rename("home_win_pct_minus_away_win_pct")
goal_adv = matches.groupby("season").apply(lambda g: (g["home_goals"] - g["away_goals"]).mean(), include_groups=False)
pd.DataFrame({"home_win% - away_win%": home_adv.round(1), "avg home goal margin": goal_adv.round(2)})

In [ ]:
fig, ax = plt.subplots()
bar_colors = ["#eb6834" if s == "2020/21" else "#2a78d6" for s in home_adv.index]
bars = ax.bar(home_adv.index, home_adv, color=bar_colors, width=0.7)
ax.axhline(0, color="#c3c2b7", linewidth=1)
ax.bar_label(bars, fmt="%.1f", padding=3, color="#52514e", fontsize=9)
ax.set_title("Home advantage: home win % minus away win % (orange = 2020/21, no crowds)")
ax.margins(y=0.12)
ax.set_ylabel("percentage points")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 4. Goals per game

In [ ]:
matches["total_goals"] = matches["home_goals"] + matches["away_goals"]
goals = matches.groupby("season")[["home_goals", "away_goals", "total_goals"]].mean().round(2)
goals

In [ ]:
fig, ax = plt.subplots()
ax.plot(goals.index, goals["home_goals"], color=COLORS["H"], linewidth=2, marker="o", markersize=6, label="Home team")
ax.plot(goals.index, goals["away_goals"], color=COLORS["A"], linewidth=2, marker="o", markersize=6, label="Away team")
for col, label in [("home_goals", "Home team"), ("away_goals", "Away team")]:
    ax.annotate(label, (len(goals) - 1, goals[col].iloc[-1]), xytext=(8, 0),
                textcoords="offset points", va="center", color="#52514e", fontsize=10)
ax.set_title("Average goals per game")
ax.set_ylabel("goals")
ax.set_ylim(0, 2)
ax.set_xlim(-0.5, len(goals) + 0.8)
ax.legend(loc="lower left", ncol=2, frameon=False)
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# How are total goals distributed? (Useful later for the Poisson goals model.)
dist = matches["total_goals"].value_counts(normalize=True).sort_index().mul(100)
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(dist.index, dist.values, color="#2a78d6", width=0.8)
ax.bar_label(bars, fmt="%.0f%%", padding=3, color="#52514e", fontsize=9)
ax.set_title("Total goals in a match")
ax.margins(y=0.1)
ax.set_xlabel("goals")
ax.set_ylabel("% of matches")
ax.set_xticks(dist.index)
plt.tight_layout()
plt.show()

In [ ]:
# Most common scorelines
matches["score"] = matches["home_goals"].astype(str) + "-" + matches["away_goals"].astype(str)
matches["score"].value_counts(normalize=True).head(10).mul(100).round(1)

## 5. A first look at the bookmakers

Bookmaker odds turn into probabilities via `1 / odds`. The three implied probabilities add up to slightly more than 1; the extra is the bookmaker's margin (the *overround*). Week 4 removes it properly. For now: how often does the bookmaker's favourite win?

In [ ]:
odds = matches[["b365_home", "b365_draw", "b365_away"]]
implied = 1 / odds
overround = implied.sum(axis=1)
print(f"Average Bet365 overround: {(overround.mean() - 1) * 100:.1f}%")

favourite = odds.idxmin(axis=1).map({"b365_home": "H", "b365_draw": "D", "b365_away": "A"})
print(f"Favourite wins: {(favourite == matches['result']).mean() * 100:.1f}% of matches")
print(f"Always pick home: {(matches['result'] == 'H').mean() * 100:.1f}% of matches")

## Takeaways

Fill these in with your own numbers and observations after running the notebook:

- Home wins / draws / away wins happen roughly __ / __ / __ % of the time.
- Home advantage in 2020/21 (no crowds) was ...
- Goals per game: ...
- The bookmaker favourite wins __ % of matches. That's a rough ceiling on how accurate a model can be, which is why Week 3 measures log loss rather than accuracy alone.